In [ ]:
### =====================================================
###  ANALISIS DE REGRESION
### =====================================================
###
install.packages("car")
install.packages("olsrr")
install.packages("tseries")
install.packages("nortest")
library(car)
library(olsrr)
library(tseries)
library(nortest)
### =====================================================
### INSTRUCCIONES DE COMO EJECUTAR CODIGO
### =====================================================

# Antes de ejecutar el código, actualizar las rutas de los train_public.csv y test_public.csv según la ubicación donde se encuentren.

### =====================================================
### 1. LECTURA Y PREPARACIN DE DATOS
### =====================================================

datos <- read.csv("train_public.csv")
test <- read.csv("test_public.csv")

datos$job_level <- as.factor(datos$job_level)
datos$sector <- as.factor(datos$sector)

test$job_level <- factor(test$job_level, levels = levels(datos$job_level))
test$sector <- factor(test$sector, levels = levels(datos$sector))

summary(datos)
str(datos)

### Verificamos si hay datos duplicados

sum(duplicated(datos$id))

### Verificamos si hay nulos

colSums(is.na(datos))


### =====================================================
### 2. MODELO 1: MODELO COMPLETO ORIGINAL
### =====================================================

## Ajustamos el modelo suponiendo que todas las variables son significativas

modelo1 <- lm(
  productivity ~ age + educ_years + experience_years +
    training_hours + remote_days + commute_min +
    urban_index + aqi + noise_db +
    sleep_hrs_week + stress_score + health_index +
    team_size + job_level + sector,
  data = datos
)

summary(modelo1)


### =====================================================
### 3. DIAGNOSTICO DEL MODELO COMPLETO
### =====================================================
## Vector de residuales
res1 <- residuals(modelo1)
##Vector de predicciones
fit1 <- fitted(modelo1)

### Verificación del supuesto de linealidad a través de los resuduales parciales

crPlots(modelo1)

### Verificacion del supuesto de normalidad

##Histograma de resiudales

hist(res1, breaks = 20, probability = TRUE,
     main = "Histograma de residuales - modelo completo",
     xlab = "Residuales")

curve(dnorm(x, mean = mean(res1), sd = sd(res1)),
      col = "red", lwd = 2, add = TRUE)

## QQ Plot

qqnorm(res1, main = "QQ-plot - modelo completo")
qqline(res1, col = "red", lwd = 2)

###Pruebas de bondad de ajuste para la normalidad

##Shapiro-Wilk Normality Test
shapiro.test(res1)
##Anderson-Darling test for normality
ad.test(res1)
##Liliiefors for normality
lillie.test(res1)
##Cramer-von Mises normality test
cvm.test(res1)

### Homocedasticidad
plot(fit1, res1,
     xlab = "Valores ajustados",
     ylab = "Residuales",
     main = "Residuales vs ajustados - modelo completo")
abline(h = 0, col = "red")

n1 <- length(res1)

g1 <- c(
  rep(1, floor(n1 / 3)),
  rep(2, floor(n1 / 3)),
  rep(3, n1 - 2 * floor(n1 / 3))
)
### Verificación de homocedasticidad
bartlett.test(res1, g1)
leveneTest(res1 ~ as.factor(g1))

### Independencia

runs.test(as.factor(res1 > 0))
### Se utilizan la prueba de rachas y Durbin-Watson
durbinWatsonTest(modelo1)

### Multicolinealidad
##Factor de Inflación de la Varianza (VIF)
vif(modelo1)
##Se construye la matriz de diseño del modelo.
X1 <- model.matrix(modelo1)
kappa(t(X1) %*% X1)
##Se seleccionan únicamente las variables numéricas
X_num1 <- datos[, c(
  "age", "educ_years", "experience_years",
  "training_hours", "remote_days", "commute_min",
  "urban_index", "aqi", "noise_db",
  "sleep_hrs_week", "stress_score",
  "health_index", "team_size"
)]

corr1 <- cor(X_num1)
round(corr1, 2)
which(abs(corr1) > 0.8 & abs(corr1) < 1, arr.ind = TRUE)

### Outliers e influencia

##Residuales studentizados
rj1 <- rstudent(modelo1)
##Valores de leverage.
h1 <- hatvalues(modelo1)
##Distancia de Cook.
cook1 <- cooks.distance(modelo1)
##Número de variables explicativas del modelo.
p1 <- length(coef(modelo1)) - 1

##Gráfico de residuales studentizados.
plot(rj1, ylim = c(-5, 5),
     xlab = "Indice",
     ylab = "Residuales Jackknife",
     main = "Residuales Jackknife - modelo completo")
abline(h = c(-3, 3), col = "red")
##Gráfico de leverage.
plot(h1,
     xlab = "Indice",
     ylab = "Leverage",
     main = "Leverage - modelo completo")
abline(h = 2 * (p1 + 1) / nrow(datos), col = "red")
##Gráfico de distancia de Cook.
plot(cook1,
     xlab = "Indice",
     ylab = "Distancia de Cook",
     main = "Cook - modelo completo")
abline(h = 4 / nrow(datos), col = "red")
which(abs(rj1) > 3)
which(h1 > 2 * (p1 + 1) / nrow(datos))
which(cook1 > 4 / nrow(datos))


### =====================================================
### 4. SELECCION DE VARIABLES SOBRE MODELO COMPLETO
### =====================================================

##Método backward
ols_step_backward_p(modelo1, prem = 0.05)
##Método forward
ols_step_forward_p(modelo1, pent = 0.05)
##Método stepwise
ols_step_both_p(modelo1, prem = 0.05, pent = 0.05)


### =====================================================
### 5. TRANSFORMACIONES PROPUESTAS
### =====================================================

datos$log_training <- log(datos$training_hours + 1)
datos$age2 <- datos$age^2
datos$commute2 <- datos$commute_min^2

test$log_training <- log(test$training_hours + 1)
test$age2 <- test$age^2
test$commute2 <- test$commute_min^2


### =====================================================
### 6. MODELO 2: MODELO TRANSFORMADO
### =====================================================
##Se ajusta un segundo modelo de regresión lineal incorporando
modelo2 <- lm(
  productivity ~ age + age2 +
    educ_years + experience_years +
    log_training + remote_days +
    commute_min + commute2 +
    urban_index + noise_db +
    sleep_hrs_week + stress_score +
    health_index + team_size +
    job_level + sector,
  data = datos
)

summary(modelo2)


### =====================================================
### 7. DIAGNOSTICO Y SELECCION DEL MODELO TRANSFORMADO
### =====================================================

##Verificacion de la lienalidad a través de los residuales
crPlots(modelo2)

ols_step_backward_p(modelo2, prem = 0.05)
ols_step_forward_p(modelo2, pent = 0.05)
ols_step_both_p(modelo2, prem = 0.05, pent = 0.05)

vif(modelo2)

X2 <- model.matrix(modelo2)
kappa(t(X2) %*% X2)


### =====================================================
### 8. MODELO 3: MODELO REDUCIDO FINAL
### =====================================================
##Se ajusta un mod red conservando únicamente los predictores más relevantes
modelo3 <- lm(
  productivity ~ educ_years + experience_years +
    log_training + remote_days +
    commute2 + noise_db +
    sleep_hrs_week + stress_score +
    health_index + team_size +
    job_level,
  data = datos
)

summary(modelo3)


### =====================================================
### 9. COMPARACION PREDICTIVA DE LOS 3 MODELOS
### =====================================================

set.seed(123)
##Se selecciona aleatoriamente el 80% de las observaciones
filas_entrenamiento <- sample(1:nrow(datos), 0.8 * nrow(datos))
##División de los datos en entrenamiento (80%)
##Validación (20%).
datos_entrenar <- datos[filas_entrenamiento, ]
datos_validar <- datos[-filas_entrenamiento, ]


### -------------------------
### Modelo 1: completo
### -------------------------
##Se ajusta nuevamente el modelo completo utilizando los datos de entrenamiento.
modelo1_entrenado <- lm(
  productivity ~ age + educ_years + experience_years +
    training_hours + remote_days + commute_min +
    urban_index + aqi + noise_db +
    sleep_hrs_week + stress_score + health_index +
    team_size + job_level + sector,
  data = datos_entrenar
)

pred1 <- predict(modelo1_entrenado, newdata = datos_validar)

mse_interno1 <- mean(residuals(modelo1_entrenado)^2)
mse_validacion1 <- mean((datos_validar$productivity - pred1)^2)
r2_ajustada1 <- summary(modelo1_entrenado)$adj.r.squared


### -------------------------
### Modelo 2: transformado
### -------------------------
##Se ajusta el modelo transformado utilizando el conjunto de entrenamiento.
modelo2_entrenado <- lm(
  productivity ~ age + age2 +
    educ_years + experience_years +
    log_training + remote_days +
    commute_min + commute2 +
    urban_index + noise_db +
    sleep_hrs_week + stress_score +
    health_index + team_size +
    job_level + sector,
  data = datos_entrenar
)

pred2 <- predict(modelo2_entrenado, newdata = datos_validar)

mse_interno2 <- mean(residuals(modelo2_entrenado)^2)
mse_validacion2 <- mean((datos_validar$productivity - pred2)^2)
r2_ajustada2 <- summary(modelo2_entrenado)$adj.r.squared


### -------------------------
### Modelo 3: reducido final
### -------------------------
##Se ajusta el modelo reducido final utilizando las variables relevantes
modelo3_entrenado <- lm(
  productivity ~ educ_years + experience_years +
    log_training + remote_days +
    commute2 + noise_db +
    sleep_hrs_week + stress_score +
    health_index + team_size +
    job_level,
  data = datos_entrenar
)

pred3 <- predict(modelo3_entrenado, newdata = datos_validar)

mse_interno3 <- mean(residuals(modelo3_entrenado)^2)
mse_validacion3 <- mean((datos_validar$productivity - pred3)^2)
r2_ajustada3 <- summary(modelo3_entrenado)$adj.r.squared


### -------------------------
### Tabla comparativa
### -------------------------
##Tabla resumen con las principales métricas
tabla_modelos <- data.frame(
  Modelo = c("Modelo completo", "Modelo transformado", "Modelo reducido final"),
  MSE_interno = c(mse_interno1, mse_interno2, mse_interno3),
  MSE_validacion = c(mse_validacion1, mse_validacion2, mse_validacion3),
  R2_ajustada = c(r2_ajustada1, r2_ajustada2, r2_ajustada3)
)

tabla_modelos


### -------------------------
### Graficas comparativas
### -------------------------

mse_matriz <- rbind(
  tabla_modelos$MSE_interno,
  tabla_modelos$MSE_validacion
)

colnames(mse_matriz) <- tabla_modelos$Modelo
rownames(mse_matriz) <- c("MSE interno", "MSE validacion")

barplot(
  mse_matriz,
  beside = TRUE,
  legend.text = TRUE,
  main = "Comparacion de MSE por modelo",
  ylab = "MSE",
  las = 2
)

barplot(
  tabla_modelos$R2_ajustada,
  names.arg = tabla_modelos$Modelo,
  main = "Comparacion de R cuadrada ajustada",
  ylab = "R cuadrada ajustada",
  las = 2
)

plot(
  datos_validar$productivity,
  pred3,
  xlab = "Productivity real",
  ylab = "Productivity predicha",
  main = "Modelo reducido final: reales vs predichos"
)

abline(0, 1, col = "red")

### =====================================================
### 10. DIAGNOSTICO FINAL DEL MODELO ELEGIDO
### =====================================================

res3 <- residuals(modelo3)
fit3 <- fitted(modelo3)

### Linealidad
crPlots(modelo3)

### Independencia
runs.test(as.factor(res3 > 0))
durbinWatsonTest(modelo3)

### Normalidad
hist(res3, breaks = 20, probability = TRUE,
     main = "Histograma de residuales - modelo final",
     xlab = "Residuales")

curve(dnorm(x, mean = mean(res3), sd = sd(res3)),
      col = "red", lwd = 2, add = TRUE)

qqnorm(res3, main = "QQ-plot - modelo final")
qqline(res3, col = "red", lwd = 2)

shapiro.test(res3)
ad.test(res3)
lillie.test(res3)
cvm.test(res3)

### Homocedasticidad
plot(fit3, res3,
     xlab = "Valores ajustados",
     ylab = "Residuales",
     main = "Residuales vs ajustados - modelo final")
abline(h = 0, col = "red")

n3 <- length(res3)

g3 <- c(
  rep(1, floor(n3 / 3)),
  rep(2, floor(n3 / 3)),
  rep(3, n3 - 2 * floor(n3 / 3))
)

bartlett.test(res3, g3)
leveneTest(res3 ~ as.factor(g3))

### Multicolinealidad final
vif(modelo3)

X3 <- model.matrix(modelo3)
kappa(t(X3) %*% X3)

X_num3 <- datos[, c(
  "educ_years",
  "experience_years",
  "log_training",
  "remote_days",
  "commute2",
  "noise_db",
  "sleep_hrs_week",
  "stress_score",
  "health_index",
  "team_size"
)]

corr3 <- cor(X_num3)
round(corr3, 2)
which(abs(corr3) > 0.8 & abs(corr3) < 1, arr.ind = TRUE)

### Outliers e influencia final
rj3 <- rstudent(modelo3)
h3 <- hatvalues(modelo3)
cook3 <- cooks.distance(modelo3)
p3 <- length(coef(modelo3)) - 1

plot(rj3, ylim = c(-5, 5),
     xlab = "Indice",
     ylab = "Residuales Jackknife",
     main = "Residuales Jackknife - modelo final")
abline(h = c(-3, 3), col = "red")

plot(h3,
     xlab = "Indice",
     ylab = "Leverage",
     main = "Leverage - modelo final")
abline(h = 2 * (p3 + 1) / nrow(datos), col = "red")

plot(cook3,
     xlab = "Indice",
     ylab = "Distancia de Cook",
     main = "Cook - modelo final")
abline(h = 4 / nrow(datos), col = "red")

which(abs(rj3) > 3)
which(h3 > 2 * (p3 + 1) / nrow(datos))
which(cook3 > 4 / nrow(datos))

influence.measures(modelo3)


### =====================================================
### 11. PREDICCIONES FINALES PARA TEST_PUBLIC
### =====================================================

pred_final <- predict(modelo3, newdata = test)

predicciones <- data.frame(
  id = test$id,
  y_hat = pred_final
)

write.csv(predicciones, "predicciones.csv", row.names = FALSE)


### =====================================================
### 12. RESUMEN FINAL
### =====================================================

summary(modelo1)
summary(modelo2)
summary(modelo3)

comparacion
vif(modelo3)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘colorspace’, ‘fracdiff’, ‘lmtest’, ‘timeDate’, ‘urca’, ‘zoo’, ‘RcppArmadillo’, ‘cowplot’, ‘Deriv’, ‘forecast’, ‘microbenchmark’, ‘rbibutils’, ‘numDeriv’, ‘doBy’, ‘SparseM’, ‘MatrixModels’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’, ‘carData’, ‘abind’, ‘Formula’, ‘pbkrtest’, ‘quantreg’, ‘lme4’


